# Bachelrorarbeit: OOD-Augmentationsstrategie bei Datenknappheit

# Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import models, transforms
from medmnist import PathMNIST, BloodMNIST
import numpy as np
import random
import copy
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Reproduzierbarkeit und Setup

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Nutze Gerät: {DEVICE}")

NUM_SAMPLES_PER_CLASS = 50
NUM_UNKNOWN_SAMPLES = 450
UNKNOWN_LABEL = 9 

# Datensätze laden

In [ ]:
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

print("Lade PathMNIST...")
train_dataset = PathMNIST(split='train', transform=data_transform, download=True, size=224)
val_dataset = PathMNIST(split='val', transform=data_transform, download=True, size=224)
test_dataset = PathMNIST(split='test', transform=data_transform, download=True, size=224)

def get_balanced_subset(dataset, n_per_class):
    labels = dataset.labels.flatten()
    indices = []
    for i in np.unique(labels):
        class_indices = np.where(labels == i)[0]
        selected_indices = np.random.choice(class_indices, n_per_class, replace=False)
        indices.extend(selected_indices)
    return Subset(dataset, indices)

# Ziel-Trainingsmenge (450 Bilder)
small_train_ds = get_balanced_subset(train_dataset, NUM_SAMPLES_PER_CLASS)

print("Lade BloodMNIST (Unknown)...")
blood_train_ds = BloodMNIST(split='train', transform=data_transform, download=True, size=224)
blood_indices = np.random.choice(len(blood_train_ds), NUM_UNKNOWN_SAMPLES, replace=False)
small_blood_ds = Subset(blood_train_ds, blood_indices)

class LabelWrapper(torch.utils.data.Dataset):
    def __init__(self, dataset, new_label):
        self.dataset = dataset
        self.new_label = new_label
    def __getitem__(self, index):
        image, _ = self.dataset[index]
        return image, np.array([self.new_label], dtype=np.int64)
    def __len__(self):
        return len(self.dataset)

unknown_ds = LabelWrapper(small_blood_ds, UNKNOWN_LABEL)

print("\n--- Daten-Setup Abgeschlossen ---")
print(f"Ziel-Trainingsdaten: {len(small_train_ds)} Bilder")
print(f"Unknown-Daten:       {len(unknown_ds)} Bilder")
print(f"Testdaten:           {len(test_dataset)} Bilder")

# Baseline Experiment

## Phase A: Optuna

In [ ]:
print("==========================================")
print("EXPERIMENT 1: BASELINE (Ohne Unknown)")
print("==========================================\n")

# --- PHASE A: Optuna ---
def objective_baseline(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32])
    
    target_labels = [train_dataset.labels[i][0] for i in small_train_ds.indices]
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    fold_accuracies = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(target_labels)), target_labels)):
        fold_train_ds = Subset(small_train_ds, train_idx)
        fold_val_ds = Subset(small_train_ds, val_idx)
        
        loader_train = DataLoader(fold_train_ds, batch_size=batch_size, shuffle=True)
        loader_val = DataLoader(fold_val_ds, batch_size=batch_size, shuffle=False)

        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, 9)
        model = model.to(DEVICE)
        
        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()

        best_acc = 0
        for epoch in range(12): # Tuning Epochen
            model.train()
            for inputs, targets in loader_train:
                inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
                optimizer.zero_grad()
                loss = criterion(model(inputs), targets)
                loss.backward()
                optimizer.step()
                
            model.eval()
            all_preds, all_targets = [], []
            with torch.no_grad():
                for inputs, targets in loader_val:
                    inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
                    _, predicted = torch.max(model(inputs), 1)
                    all_preds.extend(predicted.cpu().numpy())
                    all_targets.extend(targets.cpu().numpy())
            
            acc = accuracy_score(all_targets, all_preds)
            if acc > best_acc: best_acc = acc
        fold_accuracies.append(best_acc)
    return np.mean(fold_accuracies)

optuna.logging.set_verbosity(optuna.logging.WARNING) 
study_base = optuna.create_study(direction="maximize")
print("Suche Baseline Hyperparameter (10 Trials)...")
study_base.optimize(objective_baseline, n_trials=10)
best_lr_base = study_base.best_params["lr"]
best_bs_base = study_base.best_params["batch_size"]
print(f"Beste LR: {best_lr_base:.6f} | Beste BS: {best_bs_base}")

## Phase B: Finales Training

In [ ]:
print("\nTrainiere finales Baseline-Modell (80 Epochen)...")
final_train_loader = DataLoader(small_train_ds, batch_size=best_bs_base, shuffle=True)
final_val_loader = DataLoader(val_dataset, batch_size=best_bs_base, shuffle=False)

base_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
base_model.fc = nn.Linear(base_model.fc.in_features, 9)
base_model = base_model.to(DEVICE)

optimizer_base = optim.Adam(base_model.parameters(), lr=best_lr_base)
criterion = nn.CrossEntropyLoss()

best_base_val_acc = 0.0
best_base_weights = None

for epoch in range(80):
    base_model.train()
    for inputs, targets in final_train_loader:
        inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
        optimizer_base.zero_grad()
        loss = criterion(base_model(inputs), targets)
        loss.backward()
        optimizer_base.step()
        
    base_model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, targets in final_val_loader:
            inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
            _, predicted = torch.max(base_model(inputs), 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            
    acc = accuracy_score(all_targets, all_preds)
    if acc > best_base_val_acc:
        best_base_val_acc = acc
        best_base_weights = copy.deepcopy(base_model.state_dict())

base_model.load_state_dict(best_base_weights)

## Phase C: Test

In [ ]:
print("\nValidiere auf Test-Set...")
test_loader = DataLoader(test_dataset, batch_size=best_bs_base, shuffle=False)
base_model.eval()
all_preds, all_targets, all_probs = [], [], []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
        outputs = base_model(inputs)
        all_probs.extend(F.softmax(outputs, dim=1).cpu().numpy())
        all_preds.extend(torch.max(outputs, 1)[1].cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

# Metriken berechnen
test_acc = accuracy_score(all_targets, all_preds)
test_bal_acc = balanced_accuracy_score(all_targets, all_preds)
test_prec = precision_score(all_targets, all_preds, average='macro', zero_division=0)
test_rec = recall_score(all_targets, all_preds, average='macro', zero_division=0)
test_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
test_auc = roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro')

print("\n" + "="*50)
print("BASELINE ERGEBNISSE (TEST-SET)")
print("="*50)
print(f"Accuracy:      {test_acc*100:.2f}%")
print(f"Balanced Acc:  {test_bal_acc*100:.2f}%")
print(f"Precision:     {test_prec*100:.2f}%")
print(f"Recall:        {test_rec*100:.2f}%")
print(f"F1-Score:      {test_f1*100:.2f}%")
print(f"AUC-ROC:       {test_auc*100:.2f}%")
print("="*50)

# Speichern für den finalen Vergleich
results_baseline = {
    "Accuracy": test_acc * 100,
    "Balanced Acc": test_bal_acc * 100,
    "Precision": test_prec * 100,
    "Recall": test_rec * 100,
    "F1-Score": test_f1 * 100,
    "AUC-ROC": test_auc * 100
}

# Strategie Experiment

## Phase A: Optuna

In [ ]:
print("==========================================")
print("EXPERIMENT 2: STRATEGIE (OOD-Augmentation)")
print("==========================================\n")

# --- PHASE A: Optuna ---
def objective_strategy(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32])
    
    target_labels = [train_dataset.labels[i][0] for i in small_train_ds.indices]
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    fold_accuracies = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(target_labels)), target_labels)):
        fold_train_target = Subset(small_train_ds, train_idx)
        fold_val_target = Subset(small_train_ds, val_idx)
        fold_combined_train = ConcatDataset([fold_train_target, unknown_ds])
        
        loader_train_comb = DataLoader(fold_combined_train, batch_size=batch_size, shuffle=True)
        loader_train_targ = DataLoader(fold_train_target, batch_size=batch_size, shuffle=True)
        loader_val = DataLoader(fold_val_target, batch_size=batch_size, shuffle=False)

        # Phase 1
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, 10)
        model = model.to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()

        best_p1_acc, best_p1_weights = 0.0, None
        for epoch in range(12):
            model.train()
            for inputs, targets in loader_train_comb:
                inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
                optimizer.zero_grad()
                loss = criterion(model(inputs), targets)
                loss.backward()
                optimizer.step()
                
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for inputs, targets in loader_val:
                    inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
                    _, predicted = torch.max(model(inputs)[:, :9], 1)
                    total += targets.size(0)
                    correct += (predicted == targets).sum().item()
            acc = correct / total
            if acc > best_p1_acc:
                best_p1_acc = acc
                best_p1_weights = copy.deepcopy(model.state_dict())
                
        # Phase 2
        if best_p1_weights is not None: model.load_state_dict(best_p1_weights)
        model.fc = nn.Linear(model.fc.in_features, 9)
        model = model.to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        best_p2_acc = 0.0
        for epoch in range(12):
            model.train()
            for inputs, targets in loader_train_targ:
                inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
                optimizer.zero_grad()
                loss = criterion(model(inputs), targets)
                loss.backward()
                optimizer.step()
                
            model.eval()
            all_preds, all_targets = [], []
            with torch.no_grad():
                for inputs, targets in loader_val:
                    inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
                    _, predicted = torch.max(model(inputs), 1)
                    all_preds.extend(predicted.cpu().numpy())
                    all_targets.extend(targets.cpu().numpy())
            acc = accuracy_score(all_targets, all_preds)
            if acc > best_p2_acc: best_p2_acc = acc
        fold_accuracies.append(best_p2_acc)
    return np.mean(fold_accuracies)

study_strat = optuna.create_study(direction="maximize")
print("Suche Strategie Hyperparameter (10 Trials)...")
study_strat.optimize(objective_strategy, n_trials=10)
best_lr_strat = study_strat.best_params["lr"]
best_bs_strat = study_strat.best_params["batch_size"]
print(f"Beste LR: {best_lr_strat:.6f} | Beste BS: {best_bs_strat}")

## Phase B: Finales Training

In [ ]:
print("\nTrainiere finales Strategie-Modell...")
combined_train_ds_final = ConcatDataset([small_train_ds, unknown_ds])
final_train_loader_comb = DataLoader(combined_train_ds_final, batch_size=best_bs_strat, shuffle=True)
final_train_loader_targ = DataLoader(small_train_ds, batch_size=best_bs_strat, shuffle=True)
final_val_loader = DataLoader(val_dataset, batch_size=best_bs_strat, shuffle=False)

strat_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
strat_model.fc = nn.Linear(strat_model.fc.in_features, 10)
strat_model = strat_model.to(DEVICE)
optimizer_strat = optim.Adam(strat_model.parameters(), lr=best_lr_strat)
criterion = nn.CrossEntropyLoss()

# Phase 1 (50 Epochen)
best_final_p1_acc, best_final_p1_weights = 0.0, None
print("  -> Phase 1 (10 Klassen, 50 Epochen)...")
for epoch in range(50):
    strat_model.train()
    for inputs, targets in final_train_loader_comb:
        inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
        optimizer_strat.zero_grad()
        loss = criterion(strat_model(inputs), targets)
        loss.backward()
        optimizer_strat.step()
        
    strat_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in final_val_loader:
            inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
            _, predicted = torch.max(strat_model(inputs)[:, :9], 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
    acc = correct / total
    if acc > best_final_p1_acc:
        best_final_p1_acc = acc
        best_final_p1_weights = copy.deepcopy(strat_model.state_dict())

# Phase 2 (30 Epochen)
if best_final_p1_weights is not None: strat_model.load_state_dict(best_final_p1_weights)
strat_model.fc = nn.Linear(strat_model.fc.in_features, 9)
strat_model = strat_model.to(DEVICE)
optimizer_strat = optim.Adam(strat_model.parameters(), lr=best_lr_strat)

best_final_p2_acc, best_final_p2_weights = 0.0, None
print("  -> Phase 2 (9 Klassen, 30 Epochen)...")
for epoch in range(30):
    strat_model.train()
    for inputs, targets in final_train_loader_targ:
        inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
        optimizer_strat.zero_grad()
        loss = criterion(strat_model(inputs), targets)
        loss.backward()
        optimizer_strat.step()
        
    strat_model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, targets in final_val_loader:
            inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
            _, predicted = torch.max(strat_model(inputs), 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    acc = accuracy_score(all_targets, all_preds)
    if acc > best_final_p2_acc:
        best_final_p2_acc = acc
        best_final_p2_weights = copy.deepcopy(strat_model.state_dict())

strat_model.load_state_dict(best_final_p2_weights)

## Phase C: Test

In [ ]:
print("\nValidiere auf Test-Set...")
test_loader = DataLoader(test_dataset, batch_size=best_bs_strat, shuffle=False)
strat_model.eval()
all_preds, all_targets, all_probs = [], [], []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
        outputs = strat_model(inputs)
        all_probs.extend(F.softmax(outputs, dim=1).cpu().numpy())
        all_preds.extend(torch.max(outputs, 1)[1].cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

# Metriken berechnen
test_acc = accuracy_score(all_targets, all_preds)
test_bal_acc = balanced_accuracy_score(all_targets, all_preds)
test_prec = precision_score(all_targets, all_preds, average='macro', zero_division=0)
test_rec = recall_score(all_targets, all_preds, average='macro', zero_division=0)
test_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
test_auc = roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro')

print("\n" + "="*50)
print("STRATEGIE ERGEBNISSE (TEST-SET)")
print("="*50)
print(f"Accuracy:      {test_acc*100:.2f}%")
print(f"Balanced Acc:  {test_bal_acc*100:.2f}%")
print(f"Precision:     {test_prec*100:.2f}%")
print(f"Recall:        {test_rec*100:.2f}%")
print(f"F1-Score:      {test_f1*100:.2f}%")
print(f"AUC-ROC:       {test_auc*100:.2f}%")
print("="*50)

# Speichern für den finalen Vergleich
results_strategy = {
    "Accuracy": test_acc * 100,
    "Balanced Acc": test_bal_acc * 100,
    "Precision": test_prec * 100,
    "Recall": test_rec * 100,
    "F1-Score": test_f1 * 100,
    "AUC-ROC": test_auc * 100
}

# Finaler Vergleich

In [ ]:
print("\n" + "="*60)
print(f"{'METRIK':<15} | {'BASELINE':<15} | {'STRATEGIE':<15}")
print("-" * 60)

for key in results_baseline.keys():
    base_val = results_baseline[key]
    strat_val = results_strategy[key]
    diff = strat_val - base_val
    diff_str = f"({diff:+.2f})"
    
    print(f"{key:<15} | {base_val:>6.2f}%         | {strat_val:>6.2f}% {diff_str:>8}")

print("=" * 60)

if results_strategy['Accuracy'] > results_baseline['Accuracy']:
    print("\nFAZIT: Die OOD-Strategie hat die Performance verbessert!")
else:
    print("\nFAZIT: Die Strategie brachte in diesem Setup keine Verbesserung.")